In [56]:
import pandas as pd

In [57]:
df = pd.read_csv("orders_raw.csv")
df.head()

,order_id,customer_id,order_date,category,quantity,unit_price,discount,returned
0,4913,1116,2026-09-17,Tech,2,$115.41,20%,Yes
1,2130,96,2026-07-25,Clothing,4,$148.20,10%,No
2,3816,430,2026-09-12,Home,4,$109.99,20%,No
3,2894,329,2026-08-30,Tech,3,$101.31,10%,No
4,446,24,2026-08-17,Tech,4,$64.66,20%,No


Identify and remove duplicate orders.

In [58]:
df.drop_duplicates(inplace=True)

In [59]:
df.duplicated().sum()

np.int64(0)

Normalize categories and convert currency, percentages, dates, and return flags.

In [60]:
#Convert dates
df["order_date"] = pd.to_datetime(df["order_date"] , format="mixed", errors="coerce")
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 6000 entries, 0 to 6079
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   order_id     6000 non-null   int64         
 1   customer_id  6000 non-null   int64         
 2   order_date   5980 non-null   datetime64[ns]
 3   category     6000 non-null   object        
 4   quantity     6000 non-null   int64         
 5   unit_price   5900 non-null   object        
 6   discount     6000 non-null   object        
 7   returned     6000 non-null   object        
dtypes: datetime64[ns](1), int64(3), object(4)
memory usage: 421.9+ KB


In [61]:
#Normalize categories
df["category"] = df["category"].str.strip().str.capitalize()
df["category"].unique()

array(['Tech', 'Clothing', 'Home'], dtype=object)

In [ ]:
#Convert unit_price & discount
df["unit_price"] = df["unit_price"].replace(r"[$,.]", "", regex=True).pipe(pd.to_numeric)
df["discount"] = df["discount"].str.replace("%","").pipe(pd.to_numeric)
df

,order_id,customer_id,order_date,category,quantity,unit_price,discount,returned
0,4913,1116,2026-09-17,Tech,2,11541.0,20,Yes
1,2130,96,2026-07-25,Clothing,4,14820.0,10,No
2,3816,430,2026-09-12,Home,4,10999.0,20,No
3,2894,329,2026-08-30,Tech,3,10131.0,10,No
4,446,24,2026-08-17,Tech,4,6466.0,20,No
...,...,...,...,...,...,...,...,...
6075,5700,1008,2026-09-07,Tech,3,13839.0,10,Yes
6076,2551,968,2026-09-30,Tech,4,11151.0,0,No
6077,538,101,2026-08-06,Clothing,2,7446.0,0,Yes
6078,1221,392,2026-07-05,Clothing,2,1603.0,0,No


In [ ]:
#Convert return flags
df["returned"] = df["returned"].str.strip().str.lower().map({"yes":True,"no":False})
df

,order_id,customer_id,order_date,category,quantity,unit_price,discount,returned
0,4913,1116,2026-09-17,Tech,2,11541.0,20,True
1,2130,96,2026-07-25,Clothing,4,14820.0,10,False
2,3816,430,2026-09-12,Home,4,10999.0,20,False
3,2894,329,2026-08-30,Tech,3,10131.0,10,False
4,446,24,2026-08-17,Tech,4,6466.0,20,False
...,...,...,...,...,...,...,...,...
6075,5700,1008,2026-09-07,Tech,3,13839.0,10,True
6076,2551,968,2026-09-30,Tech,4,11151.0,0,False
6077,538,101,2026-08-06,Clothing,2,7446.0,0,True
6078,1221,392,2026-07-05,Clothing,2,1603.0,0,False


Decide how to handle missing prices, invalid dates, and nonpositive quantities.

In [66]:
df.isnull().sum()

order_id         0
customer_id      0
order_date      20
category         0
quantity         0
unit_price     100
discount         0
returned         0
dtype: int64

Separate usable rows from quarantined rows, recording rejection reasons.

Calculate net revenue and save orders_clean.csv.